# Pack an image chat for training

A LLaVA-style model trains on chats in which the user can send photos. The tokenizer
cannot turn a photo into token ids, so it leaves one placeholder id where each photo
goes, and the data pipeline swaps every placeholder for that photo's visual tokens before
the language model reads the sequence. Here you build the sequence and its labels for one
chat with two photos, then the loss the model trains on. The sizes are tiny so that every
array can be printed.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
vocab_size, width, n_visual = 40, 6, 4   # a real model: about 32,000, 4,096 and 576
IMAGE = 1   # the id the tokenizer leaves where a photo goes
END = 2     # end-of-answer marker, scored as part of the answer

embed = rng.standard_normal((vocab_size, width))   # the language model's embedding table
# The projector's output for two photos, n_visual visual tokens each. Photo 0 sits near
# 10 and photo 1 near 20, so the two are easy to tell apart in a printout.
photos = [10 + rng.standard_normal((n_visual, width)), 20 + rng.standard_normal((n_visual, width))]

chat = [
    ("user", [IMAGE, 11, 12, 13]),
    ("assistant", [21, 22, END]),
    ("user", [14, IMAGE, 15]),
    ("assistant", [23, 24, 25, END]),
]
print("token ids in the chat:", sum(len(ids) for _, ids in chat), " photos:", len(photos))

## Part 1 (coding task): The sequence and its labels

Write `pack(chat, photos, embed, image_id)`. `chat` is a list of `(role, ids)` turns, with
`role` either `"user"` or `"assistant"`. Return two arrays:

- `sequence`, one row per position the language model reads. A token id becomes its row of
  `embed`. A placeholder `image_id` becomes all the rows of the next photo: the first
  placeholder in the chat takes `photos[0]`, the second `photos[1]`, and so on.
- `labels`, one integer per row of `sequence`: the token id at every position of an
  assistant turn, and -100 at every other position, visual positions included.

In [ ]:
def pack(chat, photos, embed, image_id):
    """The rows the language model reads, and one label per row (-100 where nothing is scored)."""
    # TODO
    raise NotImplementedError

In [ ]:
seq, lab = pack(chat, photos, embed, IMAGE)
seq, lab = np.asarray(seq), np.asarray(lab)
assert seq.shape == (20, width), f"sequence shape {seq.shape}: the chat's 14 ids with each placeholder grown to {n_visual} rows make 20 rows"
assert lab.shape == (20,), f"labels shape {lab.shape}: one label per row of the sequence"
assert np.allclose(seq[0:4], photos[0]), "rows 0 to 3 should be photo 0, which replaces the first placeholder"
assert np.allclose(seq[11:15], photos[1]), "rows 11 to 14 should be photo 1: in the packed sequence the second placeholder sits at row 11"
assert np.allclose(seq[4:7], embed[[11, 12, 13]]), "rows 4 to 6 should be the embeddings of tokens 11, 12 and 13"
assert np.allclose(seq[7:10], embed[[21, 22, END]]) and np.allclose(seq[10], embed[14]), "rows 7 to 10 should be the first answer, then token 14"
assert np.allclose(seq[15], embed[15]) and np.allclose(seq[16:], embed[[23, 24, 25, END]]), "rows 15 to 19 should be token 15, then the second answer"
expected = np.array([-100] * 7 + [21, 22, END] + [-100] * 6 + [23, 24, 25, END])
assert np.array_equal(lab, expected), f"labels {lab.tolist()}: expected the answer ids at the assistant positions and -100 elsewhere"

two = [("user", [IMAGE, IMAGE, 30]), ("assistant", [31, END])]
seq2, lab2 = map(np.asarray, pack(two, photos, embed, IMAGE))
assert seq2.shape == (2 * n_visual + 3, width), f"two placeholders in one turn: shape {seq2.shape}"
assert np.allclose(seq2[:n_visual], photos[0]) and np.allclose(seq2[n_visual:2 * n_visual], photos[1]), "two placeholders in a row should take photo 0, then photo 1"
assert np.array_equal(lab2, [-100] * (2 * n_visual + 1) + [31, END]), f"labels {lab2.tolist()} for the two-photo chat"
seq3, lab3 = map(np.asarray, pack([("user", [5, 6]), ("assistant", [7, END])], [], embed, IMAGE))
assert seq3.shape == (4, width) and np.array_equal(lab3, [-100, -100, 7, END]), "a chat without photos should pack to its 4 tokens"
print("labels:", lab.tolist())
print("Part 1 passed")

## Part 2 (coding task): The next-token loss

The language model reads the packed sequence once and returns `logits`, one row per
position. Row $t$ holds its scores for every word of the vocabulary as the next token,
computed after reading positions $0$ to $t$.

Write `next_token_loss(logits, labels)`. Turn each row of scores into probabilities with a
softmax, and return the mean of $-\log P(\text{correct token})$ over every scored label: one
term for each position whose label is not -100.

In [ ]:
def next_token_loss(logits, labels):
    """Mean of -log P(correct token) over the positions whose label is not -100."""
    # TODO
    raise NotImplementedError

In [ ]:
seq, lab = map(np.asarray, pack(chat, photos, embed, IMAGE))
logits = np.random.default_rng(3).normal(0.0, 2.0, (len(lab), vocab_size))


def loop_loss(lg, lb):   # one term per scored label, from the row before it
    terms = []
    for t in range(1, len(lb)):
        if lb[t] != -100:
            row = lg[t - 1]
            terms.append(row.max() + np.log(np.exp(row - row.max()).sum()) - row[lb[t]])
    return np.mean(terms)


flat = np.zeros((len(lab), vocab_size))
assert abs(float(next_token_loss(flat, lab)) - np.log(vocab_size)) < 1e-9, "with every score equal, each prediction is uniform and the loss should be ln(vocab_size)"
got, want = float(next_token_loss(logits, lab)), loop_loss(logits, lab)
assert abs(got - want) < 1e-9, f"loss {got:.4f}, expected {want:.4f}: read again which row holds the prediction for the token at position t"
changed = logits.copy()
changed[:6] = -50.0       # these rows predict positions 1 to 6, which have no label
changed[-1] = 50.0        # the last row predicts a position after the end
assert abs(float(next_token_loss(changed, lab)) - got) < 1e-9, "rows that predict an unscored position should not change the loss"
big = float(next_token_loss(logits * 400, lab))
assert np.isfinite(big), "the loss should stay finite for scores in the hundreds"
assert abs(big - loop_loss(logits * 400, lab)) < 1e-6, "wrong loss for scores in the hundreds"
print(f"loss with random scores: {got:.4f} over {int((lab[1:] != -100).sum())} scored tokens")
print("Part 2 passed")

## Solution

### Part 1: The sequence and its labels

In [ ]:
def pack(chat, photos, embed, image_id):
    """The rows the language model reads, and one label per row (-100 where nothing is scored)."""
    rows, labels = [], []
    photo_index = 0
    for role, ids in chat:
        for token in ids:
            if token == image_id:
                block = photos[photo_index]
                photo_index += 1
                rows.append(block)
                labels += [-100] * len(block)
            else:
                rows.append(embed[token][None, :])
                labels.append(token if role == "assistant" else -100)
    return np.concatenate(rows), np.array(labels)

**Why it works.** Walking the turns in order and appending as you go keeps every row next
to its label, so each photo lands where its placeholder was and nothing has to be moved.
Building `embed[ids]` first and then splicing photos in by the placeholders' positions in
`ids` goes wrong from the second photo on: the first photo adds `n_visual - 1` rows, so every
later position in the packed sequence is that much further down than in `ids`. The visual
rows get -100 because they are vectors with no token id to predict, and the user's rows
because the model is trained to write the assistant's side, not to guess what users ask.

### Part 2: The next-token loss

In [ ]:
def next_token_loss(logits, labels):
    """Mean of -log P(correct token) over the positions whose label is not -100."""
    logits = np.asarray(logits, dtype=float)
    labels = np.asarray(labels)
    predictors, targets = logits[:-1], labels[1:]      # row t predicts position t + 1
    scored = targets != -100
    rows, answer = predictors[scored], targets[scored]
    top = rows.max(axis=1)
    log_total = top + np.log(np.exp(rows - top[:, None]).sum(axis=1))
    return float(np.mean(log_total - rows[np.arange(len(rows)), answer]))

**Why it works.** Row $t$ has read positions $0$ to $t$, so the token it is scored on sits
at $t + 1$: pair `logits[:-1]` with `labels[1:]`, and the last row, which predicts past the
end, drops out. The lesson's stand-in scored row $t$ against label $t$, which is right only
when the labels have already been moved one step. Here the 7 scored labels are the two
answers, end markers included, and the rows that predict them are the last user position
before each answer and the answer positions themselves. Subtracting each row's maximum
before `np.exp` leaves the log of the sum unchanged and keeps it from overflowing, which is
what the scores in the hundreds test.